In [0]:
 
import requests
import time
import uuid
import logging

from pyspark.sql.functions import current_timestamp, current_date, lit
from pyspark.sql.types import StructType


BASE_URL = "https://test-api-2-r15g.onrender.com"

crm_routes = ["customers", "products", "sales"]
erp_routes = ["erp_customers", "erp_locations", "erp_categories"]

BRONZE_DB = "sqldatawarehouse.bronze"
AUDIT_TABLE = f"{BRONZE_DB}.ingestion_audit"

batch_id = str(uuid.uuid4())


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_ingestion")


In [0]:


def fetch_api_data(url, retries=3):
    for i in range(retries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()

            if isinstance(data, list):
                return data
            else:
                logger.warning(f"Unexpected response format: {url}")
                return None

        except Exception as e:
            logger.warning(f"Attempt {i+1} failed for {url}: {e}")
            time.sleep(2 ** i)

    return None



In [0]:

from datetime import datetime

def log_audit(route, table_name, row_count, status):
    audit_df = spark.createDataFrame([(
        route,
        table_name,
        row_count,
        batch_id,
        status,
        datetime.now()
    )], [
        "route",
        "table_name",
        "row_count",
        "batch_id",
        "status",
        "ingestion_time"
    ])

    audit_df.write.mode("append").saveAsTable(AUDIT_TABLE)



In [0]:
from pyspark.sql.types import StructType, StructField, StringType

def normalize_records(records):
    """
    Ensures all dicts have same keys and values are string-safe
    """
    if not records:
        return records

    # collect all possible keys
    all_keys = set()
    for r in records:
        all_keys.update(r.keys())

    normalized = []
    for r in records:
        new_r = {}
        for k in all_keys:
            val = r.get(k, None)

            # force everything to string OR None
            if isinstance(val, (dict, list)):
                new_r[k] = str(val)
            else:
                new_r[k] = val

        normalized.append(new_r)

    return normalized

In [0]:
def get_max_timestamp(table_name, timestamp_column="ingestion_time"):
    """
    Get the maximum timestamp from an existing bronze table for incremental loading.
    Returns None if table doesn't exist or has no data.
    """
    try:
        # Check if table exists
        if spark.catalog.tableExists(table_name):
            # Check if the table has any data
            df = spark.table(table_name)
            
            # Try to get max timestamp
            max_ts = df.selectExpr(f"max({timestamp_column}) as max_ts").first()["max_ts"]
            
            if max_ts:
                logger.info(f"Found max timestamp in {table_name}: {max_ts}")
                return max_ts
            else:
                logger.info(f"Table {table_name} exists but has no data")
                return None
        else:
            logger.info(f"Table {table_name} does not exist yet")
            return None
            
    except Exception as e:
        logger.warning(f"Could not get max timestamp from {table_name}: {e}")
        return None

In [0]:

def ingest_api_data(route, table_prefix):

    url = f"{BASE_URL}/api/{route}"
    table_name = f"{BRONZE_DB}.{table_prefix}_{route}"

    try:
        records = fetch_api_data(url)

        if not records:
            logger.info(f"No data for {route}")
            log_audit(route, table_name, 0, "EMPTY")
            return

        # Create DataFrame
        records = normalize_records(records)
        df = spark.createDataFrame(records)
        
        # Incremental load for crm_customers only
        if table_prefix == "crm" and route == "customers":
            max_ingestion_time = get_max_timestamp(table_name)
            
            if max_ingestion_time and "cst_updated_date" in df.columns:
                # Filter to only get records updated after the last ingestion
                from pyspark.sql.functions import col, expr
                
                # Convert updated column to timestamp using try_to_timestamp (returns NULL for invalid values)
                df = df.withColumn("updated_ts", expr("try_to_timestamp(cst_updated_date)"))
                
                original_count = df.count()
                df = df.filter(col("updated_ts") > lit(max_ingestion_time))
                filtered_count = df.count()
                
                logger.info(f"Incremental load: {original_count} total records, {filtered_count} new records (updated > {max_ingestion_time})")
                
                # Drop the temporary timestamp column
                df = df.drop("updated_ts")
                
                if filtered_count == 0:
                    logger.info(f"No new records for {route} since last ingestion")
                    log_audit(route, table_name, 0, "NO_NEW_RECORDS")
                    return

        # Enrich metadata (Bronze standard) - using withColumns for better performance
        df = df.withColumns({
            "ingestion_time": current_timestamp(),
            "ingestion_date": current_date(),
            "batch_id": lit(batch_id),
            "source_system": lit(table_prefix),
            "source_route": lit(route)
        })

        # Get row count before write (action inside try block)
        row_count = df.count()

        # Write to Delta (Bronze = append-only)
        (
            df.write
            .format("delta")
            .mode("append")
            .partitionBy("ingestion_date")
            .saveAsTable(table_name)
        )

        logger.info(f"SUCCESS → {table_name} | rows={row_count}")

        log_audit(route, table_name, row_count, "SUCCESS")

    except Exception as e:
        logger.error(f"FAILED → {route}: {e}")
        log_audit(route, table_name, 0, f"FAILED: {str(e)[:200]}")



In [0]:

for route in crm_routes:
    ingest_api_data(route, "crm")

for route in erp_routes:
    ingest_api_data(route, "erp")

In [0]:
# %sql
# drop table sqldatawarehouse.bronze.api_customers;
# drop table sqldatawarehouse.bronze.crm_customers;
# drop table sqldatawarehouse.bronze.crm_products;
# drop table sqldatawarehouse.bronze.crm_sales;
# drop table sqldatawarehouse.bronze.erp_erp_categories;
# drop table sqldatawarehouse.bronze.erp_erp_customers;
# drop table sqldatawarehouse.bronze.erp_erp_locations;

In [0]:
%sql
select * from sqldatawarehouse.bronze.crm_customers where cst_updated_date = '2026-07-10';
